In [ ]:
import numpy as np
import pandas as pd
import nltk
import string
from matplotlib import pyplot as plt

In [ ]:
df = pd.read_csv('IMDB Dataset.csv')
df.head()

In [ ]:
print("Sentiments empty :",df['sentiment'].isnull().sum())

In [ ]:
if df['sentiment'].dtype == object :
  df['sentiment'] = df['sentiment'].str.lower().map({'positive':1,'negative':0})

In [ ]:
nltk.download('stopwords')
from nltk.corpus import stopwords

In [ ]:
stop_words = set(stopwords.words('english'))

In [ ]:
def clean_text(text):
  text = text.lower()
  text = text.translate(str.maketrans('','',string.punctuation))
  tokens = text.split()
  tokens = [word for word in tokens if word not in stop_words]
  text = ' '.join(tokens)
  return text

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df['review'],df['sentiment'],test_size=0.2,random_state=42)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vectorize = CountVectorizer(
preprocessor = clean_text,stop_words = "english",ngram_range=(1,2),min_df=2,max_df=0.95)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [ ]:
pipe = Pipeline([('vec',vectorize),('log',LogisticRegression(max_iter=1000,class_weight='balanced'))])
pipe.fit(X_train,y_train)

In [ ]:
from sklearn.metrics import (accuracy_score,f1_score,recall_score,precision_score,classification_report,confusion_matrix)
y_pred = pipe.predict(X_test)
print("Accuracy Score :",accuracy_score(y_test,y_pred))
print("F1 Score :",f1_score(y_test,y_pred))
print("Recall Score :",recall_score(y_test,y_pred))
print("Precision Score :",precision_score(y_test,y_pred))
print("Classification Report :\n",classification_report(y_test,y_pred))

# Bonus :

In [ ]:
feature_names = vectorize.get_feature_names_out()
coeff = pipe.named_steps['log'].coef_[0]
top_p = coeff.argsort()[-10:]
top_n = coeff.argsort()[:10]
print("TOP POSITIVE WORDS :")
for i in top_p:
  print(feature_names[i])
print("TOP NEGATIVE WORDS :")
for i in top_n:
  print(feature_names[i])

In [ ]:
from collections import Counter

positive_C = Counter(feature_names[top_p]).most_common(10)
negative_C = Counter(feature_names[top_n]).most_common(10)

p_w , p_f = zip(*positive_C)
n_w , n_f = zip(*negative_C)

In [ ]:
plt.figure(figsize=(10,5))
plt.barh(p_w,p_f,color='green',label='Positive')
plt.title('Top 10 Positive')
plt.xlabel('Frequency')
plt.gca().invert_yaxis()
plt.show()

plt.figure(figsize=(10,5))
plt.barh(n_w,n_f,color='red',label='Negative')
plt.title('Top 10 Negative')
plt.xlabel('Frequency')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
from sklearn.naive_bayes import MultinomialNB


In [ ]:
mmodel = MultinomialNB()

X_train_vec = vectorize.transform(X_train)
X_test_vec = vectorize.transform(X_test)

mmodel.fit(X_train_vec, y_train)

y_pred_nb = mmodel.predict(X_test_vec)

accuracy_nb = accuracy_score(y_test, y_pred_nb)
print("Naive Bayes Accuracy:", accuracy_nb)
print("Logistic Regression Accuracy:",accuracy_score(y_test,y_pred))